# S9 - Sistemas expertos (Pagarium)

**Objetivo (RA5-a/c):** entender la **anatomía** de un sistema experto y el ciclo *reconocer-resolver-actuar* construyendo un micro-motor, viendo **por qué falla** y comparándolo con `experta` y CLIPS.

> Hilo conductor: **Pagarium**, una pasarela de pagos que decide si aprueba o rechaza una transacción.

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre (una *variable*) |
| `{ "clave": "valor", ... }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: varios valores ordenados |
| `def nombre(...):` | Crea una **función** (bloque de código con nombre) |
| `class Nombre:` | Crea una **clase** (plantilla de objetos) |
| `self` | Dentro de una clase, se refiere "a este objeto" |
| `lambda x: ...` | Una función corta escrita en una sola línea |
| `if ...:` / `for ... in ...:` / `while ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

**No necesitas saber programar de memoria**: lee los comentarios `#` y ejecuta las celdas de arriba a abajo.

## 1. Un micro-motor (lo construimos paso a paso)

Vamos a montarlo **pieza a pieza**. Completa cada `# TODO` y ejecuta. Al final verás **por qué falla**: dispara todas las reglas que encajan sin elegir, y produce dos decisiones contradictorias.

### Paso 1 · La plantilla (clase) y sus tres almacenes

Una **clase** (`class`) es una plantilla que crea objetos. `__init__` se ejecuta al crear el objeto y prepara los datos. El motor guarda: **hechos** (diccionario), **reglas** (lista) y **traza** (lista, el orden de disparo).

In [ ]:
class MicroMotor:
    def __init__(self):
        # TODO: crea tres almacenes con self (el punto se lee "de este objeto"):
        #   self.hechos = {}   # diccionario para los hechos
        #   self.reglas = []   # lista para las reglas
        #   self.traza  = []   # lista para el orden de disparo
        pass   # borra esta línea cuando completes el TODO

### Paso 2 · Añadir hechos (método `hecho`)

Un **hecho** es una pareja nombre=valor (p. ej. `importe=200`). `hecho(**h)` recibe varias parejas y las guarda en el diccionario con `.update()`.

In [ ]:
    def hecho(self, **h):        # **h recoge las parejas nombre=valor
        # TODO: añade esas parejas al diccionario self.hechos
        #       (pista: el método se llama .update(h))
        pass

### Paso 3 · Añadir reglas (método `regla`)

Una **regla** guarda nombre, prioridad, condición y acción. La metemos en la lista `self.reglas` con `.append()`, como una tupla `(nombre, prioridad, condicion, accion)`.

In [ ]:
    def regla(self, nombre, prioridad, condicion, accion):
        # TODO: añade (nombre, prioridad, condicion, accion) a self.reglas
        #       (pista: usa self.reglas.append(...))
        pass

### Paso 4 · El ciclo reconocer-actuar (método `run`)

Repetimos mientras quede algo por disparar: recorremos las reglas y, si su **condición** se cumple, ejecutamos su **acción**. De momento **no resolvemos conflictos**.

In [ ]:
    def run(self):
        pendientes = True
        # TODO: mientras pendientes sea True:
        #   pon pendientes = False
        #   recorre con for cada (nombre, prioridad, condicion, accion) de self.reglas:
        #     si nombre ya está en self.traza -> continue
        #     si condicion(self.hechos) se cumple:
        #        apunta nombre en self.traza, ejecuta accion(self.hechos), pon pendientes = True
        #   al salir, devuelve self.hechos
        pass

### Paso 5 · Crear el motor y probarlo

Juntamos las piezas. Las **lambdas** son funciones de una línea: reciben los hechos y devuelven `True`/`False`.

In [ ]:
motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)   # hechos del pago

motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,      # condición: importe < 500
            lambda h: h.update(decision="aprobar"))                  # acción: decide "aprobar"
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,  # condición: reintentos >= 4
            lambda h: h.update(decision="rechazar"))                 # acción: decide "rechazar"

print(motor.run())     # -> {'importe': 200, ..., 'decision': 'rechazar'}  ¡y antes pasó por 'aprobar'!
print(motor.traza)     # -> ['aprobar_pequena', 'rechazar_reintentos']

## 2. Actividad — corregir el conflicto con `salience`

Modifica `run()` para que, cuando varias reglas encajen, se dispare **solo la de mayor prioridad** (eso es `salience`). Comprueba que ya no hay contradicción.

> Pista: en vez de recorrer `for` todas las reglas, agrupa las que encajan y elige la de mayor `prioridad` con `max(...)`.

In [ ]:
# TODO: reescribe run() con resolución de conflictos por prioridad
class MicroMotor2:
    ...

## 3. La misma política con `experta`

`experta` es una librería que **ya trae** el ciclo reconocer-resolver-actuar y la `salience`. Aquí `aprobar` exige `importe < 500` **y** `n_intentos < 4`, de modo que las dos reglas no pueden encajar a la vez.

> **Truco de `experta`:** las condiciones se escriben con `P(...)` (predicado), no con `lambda`; y **no** uses `self` dentro de `P(...)`.

In [ ]:
%pip install experta

# Parche obligatorio en Python 3.10+: la librería usa collections.Mapping, que ya no existe.
import collections, collections.abc
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *                  # importa Fact, Rule, KnowledgeEngine, etc.

class Pagarium(KnowledgeEngine):       # una clase que hereda del motor de experta
    @DefFacts()                        # decorador: hechos que se cargan al empezar
    def inicio(self):
        yield Fact(accion="evaluar")   # hecho inicial

    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)   # salience = prioridad
    def aprobar(self):                 # esta regla solo se dispara si importe<500 Y n_intentos<4
        self.declare(Fact(decision="aprobar"))    # .declare añade un hecho nuevo

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)  # más prioridad
    def rechazar(self):
        self.declare(Fact(decision="rechazar"))

motor = Pagarium(); motor.reset()      # crea el motor y lo deja listo
motor.declare(Fact(importe=200, n_intentos=5))   # hechos del pago
motor.run()                            # ejecuta el ciclo de inferencia
print([dict(f) for f in motor.facts.values() if "decision" in f])   # muestra solo los hechos con 'decision'

## 4. La misma política en CLIPS (`clipspy`)

`clipspy` es el enlace de Python con **CLIPS 6.4** (producción). Las reglas se escriben en una sintaxis LISP con paréntesis, y la prioridad se marca con `(declare (salience 30))`.

In [ ]:
%pip install clipspy
import clips

env = clips.Environment()              # crea el entorno de CLIPS
# build(...) define UNA regla cada vez (no dos a la vez). Cada llamada añade una regla.
env.build("(defrule aprobar (importe ?i&:(< ?i 500)) (n_intentos ?n&:(< ?n 4)) => (assert (decision aprobar)))")
env.build("(defrule rechazar (declare (salience 30)) (n_intentos ?n&:(>= ?n 4)) => (assert (decision rechazar)))")
#   (defrule nombre   condiciones-con-patrones       =>   acciones(assert añade hechos))

env.assert_string("(importe 200)")     # añade un hecho: importe 200
env.assert_string("(n_intentos 5)")    # añade otro hecho
env.run()                              # ejecuta
print([str(f) for f in env.facts()])   # lista todos los hechos finales

### Otro dominio: control de calefacción

El **mismo** micro-motor sirve para cualquier dominio; solo cambian las reglas y los hechos. Aquí controla una calefacción.

In [ ]:
# El mismo micro-motor, otro dominio: calefacción
motor = MicroMotor()
motor.hecho(temperatura=18, hora=23, ventana_abierta=False)   # hechos del ambiente

motor.regla("frio", 10, lambda h: h["temperatura"] < 20, lambda h: h.update(encender=True))   # si hace frío, encender
motor.regla("noche", 10, lambda h: h["hora"] >= 22, lambda h: h.update(modo="eco"))           # si es de noche, modo eco
motor.regla("ventilar", 5, lambda h: h["ventana_abierta"], lambda h: h.update(encender=False))  # ventana abierta: no encender

print(motor.run())    # hechos finales
print(motor.traza)    # reglas disparadas

### Otro dominio: clasificación de animales

`experta` con reglas de un zoólogo. Fíjate en el **encadenamiento hacia delante**: un hecho (`pelo`) dispara una regla que crea otro hecho (`mamifero`), y ese dispara la siguiente.

In [ ]:
class Animales(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(analizar=True)      # arranque

    @Rule(Fact(analizar=True), salience=10)
    def datos(self):                    # carga los datos del animal a clasificar
        self.declare(Fact(pelo=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))

    @Rule(Fact(pelo=True))              # si tiene pelo -> es mamífero (regla intermedia)
    def mamifero(self):
        self.declare(Fact(mamifero=True))

    @Rule(Fact(mamifero=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    def guepardo(self):                 # todas las pistas encajan -> conclusión
        print("IDENTIFICADO: guepardo")

engine = Animales(); engine.reset(); engine.run()   # ejecuta: se ve el encadenamiento

### Más reglas en CLIPS

Dos reglas que deciden la tarifa según el importe: a partir de 100 €, tarifa premium.

In [ ]:
env = clips.Environment()
# (?i&:(< ?i 100))  significa: la variable ?i debe cumplir (< ?i 100) = "?i < 100"
env.build("(defrule pequena (importe ?i&:(< ?i 100)) => (assert (tarifa basica)))")
env.build("(defrule grande (importe ?i&:(>= ?i 100)) => (assert (tarifa premium)))")
env.assert_string("(importe 150)")   # importe 150 -> cumple la regla 'grande'
env.run()
print([str(f) for f in env.facts()])

**Cierre:** compara la traza de los tres motores y explica en 3 líneas qué papel juega la **resolución de conflictos**.